# Reference Checker

Hypothesis: a hallucinated reference will (a) produce a title of a paper that doesn't exist, (b) make up authors for paper titles that do exist.

This script pulls the references section out of a PDF, pulls the references, and attempts to verify each title against Semantic Scholar.
If it finds the title in Semantic Scholar, it then attempts to verify each author, according to Semantic Scholar's records, appears in the reference.

It is really hard to deal with all reference formats, but also idosycracies and casual mistakes. This script attempts to find a title by looking for a span of dictionary-recognizable words under the assumption that names do not make for long spans of dictionary-recognizable words. The algorithm is allowed to see a maximum of one non-dictionary token in a row before concluding that a span is not the title. If there is more than one span that could be a title, it will pick the longest.

This algorithm will produce a lot of false alarms where it simply fails to pull the title out of reference.

**To Use:**

Run `check_refs(filepath)`

**Notes:**

- Some PDFs that will be reviewed have line numbers. The line numbers get interjected into the middle of text spans. the `pdf_has_line_numbers=True` option will remove all numbers from references. This shouldn't matter if the pdf has line numbers or not because the algorithm should already ignore dates.

- Add words that are not in the dictionary to `CUSTOM_VOCAB`.

- Add words that you expect never to be in a paper title to `FILTER`.

- Doesn't handle names with accent marks.

# Install Packages

In [2]:
!pip install pymupdf4llm

In [169]:
!pip install spacy

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 22.5 MB/s  0:00:00 eta 0:00:01
Using cached catalogue-2.0.10-py3-none-any.whl (17 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.9 MB/s  0:00:00
Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl (29 kB)
Using cached spacy_loggers-1.0.5-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65

In [64]:
!pip install PyEnchant

# Imports

In [202]:
import pymupdf4llm
import re
import enchant
import spacy
import unidecode
import string
import requests
import time
from functools import reduce
# Load the English language model
NLP = spacy.load("en_core_web_sm")
DICTIONARY = enchant.Dict("en_US")

# Globals

In [203]:
SEMANTIC_SCHOLAR_URL = 'https://api.semanticscholar.org/graph/v1/paper/search/match?query=' 

In [204]:
CUSTOM_VOCAB = ['ai', 'xai', 'operationalizing', 'seamful', 'llm', 'llms']

In [205]:
FILTER = ['proceedings', 'conference',
          'NY', 'USA']

# Helpers

In [7]:
def tokenize(text):
    return re.findall(r"\w+|[^\w\s-]", unidecode.unidecode(text))

In [12]:
def detokenize(tokens):
    result = ''
    for token in tokens:
        if token in string.punctuation:
            result = result + token
        else:
            result = result + ' ' + token
    return result.strip()

In [8]:
def is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

In [189]:
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

In [138]:
def does_contain(word_list, targets):
    return reduce(lambda a, b: a | b, 
                  map(lambda t: t.lower() in [w.lower() for w in word_list], 
                      targets))

In [195]:
def remove_hanging_punctuation(word_list):
    if word_list[-1] in string.punctuation:
        return word_list[0:-1]
    else:
        return word_list

# Get Reference Section 

In [169]:
def get_ref_section(md_text):
    m = re.search(r'# [0-9 ]*\*\*References\*\*([a-zA-Z0-9 \(\)\.\,\;]*)', md_text)
    if m is not None:
        after = md_text[m.span()[1]:]
        m = re.search(r'# \*\*', after)
        if m is not None:
            return after[0:m.span()[0]].strip()
        else:
            return after
    else:
        return None

# Extract Title

Each reference is on its own line. Each reference is further broken into a list of tokens (words, punctuation)

In [196]:
def extract_title(tokens, verbose=False):
    candidates = [] # Candidate titles, longest preferred
    title = [] # Current title we are building
    skip = False # We get one skip in a row
    # Iterate through tokens
    for token in tokens:
        # Part of speech tagging
        doc = nlp(token)
        if verbose:
            print(">>", token)
        # If we get a hash or star, we crash out
        # if token in ['#', '*']:
        #     return None
        # If we get certain punctuation we finish the title building
        if token in ['.', ';', '[', ']', '(', ')']:
            if verbose:
                print("PUNCT")  
                print("TITLE=", title)  
            # If title is 4 or more, then we keep it
            if len(title) > 3:
                if verbose:
                    print("CANDIDATE FOUND")
                candidates.append(title)
                title = []
                skip = False
            # If title is less than 4 we throw it out
            else:
                if verbose:
                    print("NOT A CANDIDATE")
                title = []
                skip = False
        # We cannot start a title with , or and or :
        elif (token == ',' or token == 'and' or token == ':') and len(title) == 0:
            if verbose:
                print("START WITH COMMA OR AND")
            title = []
            skip = False
        # We found something that is in the dictionaries, and is length greater than 1 (unless I or A) and is not "and"
        elif (DICTIONARY.check(token) or token.lower() in CUSTOM_VOCAB) and (len(token) > 1 or token == 'I' or token.lower() == 'a') and token.lower != 'and':
            title.append(token)
            skip = False
            if verbose:
                print("TOK")
        # Whatever remains is probably okay, but we use a skip
        elif not skip:
            skip = True
            title.append(token)
            if verbose:
                print("TOK+SKIP")
        # If we are here, we are on our second skip, give up on this
        else:
            title = []
            skip = False
            if verbose:
                print("SKIP") 
    # Now we filter out candidates
    filtered_candidates = list(filter(lambda title: not does_contain(title, FILTER), candidates)) 
    # remove orphaned punctuation
    filtered_candidates = list(map(lambda title: remove_hanging_punctuation(title),
                                   filtered_candidates))
    if len(filtered_candidates) > 0:
        sorted_candidates = sorted(filtered_candidates, key=len, reverse=True)
        return list(filter(lambda token: not is_number(token), sorted_candidates[0]))
    else:
        return None

# Access Semantic Scholar

An `author` is a json structure.

In [64]:
def get_authors_from_semantic_scholar(title):
    url = SEMANTIC_SCHOLAR_URL + title
    query_params = {"fields": "title,authors"}
    headers = {}
    response = requests.get(url, params=query_params, headers=headers)
    if response.status_code == 200:
        response_data = response.json()
        return response_data['data'][0]['authors']
    else:
        return None

In [207]:
def check_author(ref, author):
    last_name = author['name'].split()[-1]
    return last_name.lower() in ref.lower()

def check_authors(ref, authors):
    success = True
    for author in authors:
        if not check_author(ref, author):
            print("AUTHOR", author['name'], "NOT FOUND")
            success = False
    return success

# Check References

`pdf_has_line_numbers=True` will remove all numbers from each reference line.

In [200]:
def check_refs(filename, sleep=10, pdf_has_line_numbers = False):
    # Convert PDF to markdown
    print("Converting PDF to markdown...")
    md_text = pymupdf4llm.to_markdown(filename)
    # get references section
    refs = get_ref_section(md_text).strip().replace('_', '').split('\n\n')
    # Each reference should now be a separate string in a list
    print("Checking", len(refs), "refs...")
    for n, ref in enumerate(refs):
        print(n)
        if pdf_has_line_numbers:
            ref = remove_numbers(ref)
        title = extract_title(tokenize(ref))
        if title is not None and len(title) > 0:
            title = detokenize(title)
            print(title)
            # search semantic scholar and bring back a data record including authors
            authors = get_authors_from_semantic_scholar(title)
            if authors is not None:
                print("FOUND in Semantic Scholar")
                # check authors
                if check_authors(ref, authors):
                    print("OK")
            else:
                print("NOT FOUND in Semantic Scholar")
        else:
            # No title found
            print(ref)
            print('NO TITLE FOUND')
        print('\n')
        time.sleep(sleep)
        

# Run Me

In [212]:
check_refs("tests/2432.pdf", pdf_has_line_numbers = True)

Converting PDF to markdown...
=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

# For Testing

In [182]:
filename = "tests/2432.pdf"
md_text = pymupdf4llm.to_markdown(filename)

=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [185]:
md_text

'Under review as a conference paper at COLM 2026 \n\n## **Demystifying Reinforcement Learning Post-Training of Language Models** \n\n## **Anonymous authors** \n\nPaper under double-blind review \n\n## **Abstract** \n\n1 Reinforcement learning (RL) post-training has emerged as a powerful 2 framework for enhancing the capabilities of large language models (LLMs), 3 enabling impressive reasoning, math, and coding capabilities. Yet for many 4 researchers and practitioners, the principles behind classical RL remain a 5 “black box”. In this work, we deconstruct the RL post-training algorithm, 6 investigating each step to clarify what is actually happening beneath the 7 surface. By isolating the mechanics of RL with Verifiable Rewards in a 8 controlled, simplified environment, we examine how RL outcomes are 9 shaped by the base model’s prior distribution, the granularity of the reward 10 signal, the diversity of the prompt distribution, and model scale. We use 11 the entropy of the policy’s o

In [186]:
refs = get_ref_section(md_text).strip().replace('_', '').split('\n\n')
refs

['- 363 Yuntao Bai, Saurav Kadavath, Sandipan Kundu, Amanda Askell, Jackson Kernion, Andy 364 Jones, Anna Chen, Anna Goldie, Azalia Mirhoseini, Cameron McKinnon, Carol Chen, 365 Catherine Olsson, Christopher Olah, Danny Hernandez, Dawn Drain, Deep Ganguli, 366 Dustin Li, Eli Tran-Johnson, Ethan Perez, Jamie Kerr, Jared Mueller, Jeffrey Ladish, Joshua 367 Landau, Kamal Ndousse, Kamile Lukosuite, Liane Lovitt, Michael Sellitto, Nelson Elhage, 368 Nicholas Schiefer, Noemi Mercado, Nova DasSarma, Robert Lasenby, Robin Larson, Sam 369 Ringer, Scott Johnston, Shauna Kravec, Sheer El Showk, Stanislav Fort, Tamera Lanham, 370 Timothy Telleen-Lawton, Tom Conerly, Tom Henighan, Tristan Hume, Samuel R. Bowman, 371 Zac Hatfield-Dodds, Ben Mann, Dario Amodei, Nicholas Joseph, Sam McCandlish, Tom 372 Brown, and Jared Kaplan. Constitutional ai: Harmlessness from ai feedback, 2022. URL 373 https://arxiv.org/abs/2212.08073. ',
 '- 374 Fan Chen, Audrey Huang, Noah Golowich, Sadhika Malladi, Adam Block, 

In [210]:
ref = remove_numbers(refs[4])
ref

'-  Paul Christiano, Jan Leike, Tom B. Brown, Miljan Martic, Shane Legg, and Dario Amodei.  Deep reinforcement learning from human preferences, . URL https://arxiv.org/  abs/.. '

In [211]:
extract_title(tokenize(ref), verbose=True)

>> Paul
TOK
>> Christiano
TOK+SKIP
>> ,
SKIP
>> Jan
TOK
>> Leike
TOK+SKIP
>> ,
SKIP
>> Tom
TOK
>> B
TOK+SKIP
>> .
PUNCT
TITLE= ['Tom', 'B']
NOT A CANDIDATE
>> Brown
TOK
>> ,
TOK+SKIP
>> Miljan
SKIP
>> Martic
TOK+SKIP
>> ,
SKIP
>> Shane
TOK
>> Legg
TOK+SKIP
>> ,
SKIP
>> and
START WITH COMMA OR AND
>> Dario
TOK
>> Amodei
TOK+SKIP
>> .
PUNCT
TITLE= ['Dario', 'Amodei']
NOT A CANDIDATE
>> Deep
TOK
>> reinforcement
TOK
>> learning
TOK
>> from
TOK
>> human
TOK
>> preferences
TOK
>> ,
TOK+SKIP
>> .
PUNCT
TITLE= ['Deep', 'reinforcement', 'learning', 'from', 'human', 'preferences', ',']
CANDIDATE FOUND
>> URL
TOK
>> https
TOK+SKIP
>> :
SKIP
>> /
TOK+SKIP
>> /
SKIP
>> arxiv
TOK+SKIP
>> .
PUNCT
TITLE= ['arxiv']
NOT A CANDIDATE
>> org
TOK
>> /
TOK+SKIP
>> abs
TOK
>> /
TOK+SKIP
>> .
PUNCT
TITLE= ['org', '/', 'abs', '/']
CANDIDATE FOUND
>> .
PUNCT
TITLE= []
NOT A CANDIDATE


['Deep', 'reinforcement', 'learning', 'from', 'human', 'preferences']